# Full AIDE — train Tiny-GenImage combined, test Tiny + CommFor

Notebook này huấn luyện **toàn bộ kiến trúc AIDE**:

`ảnh → 4 patch DCT (low₁, high₁, low₂, high₂) → 30 SRM → 2×ResNet50 → forensic 2048-D`

`ảnh raw → OpenCLIP ConvNeXt-XXLarge (frozen) → projection 256-D`

`concat 2304-D → MLP 1024 → Real/Fake`

Thiết lập thí nghiệm:

- Train: toàn bộ các generator trong split train của Tiny-GenImage (**combined**).
- Model selection: validation stratified tách từ train; CommFor không tham gia chọn checkpoint.
- Test 1: split validation/test chính thức của Tiny-GenImage, có metric theo từng generator.
- Test 2: đúng 1.000 ảnh CommFor gồm 100 real (20 từ mỗi nguồn LAION/RAISE/ImageNet/FFHQ/COCO) và 900 fake (100 ảnh cho mỗi generator unseen).
- Checkpoint `latest` hỗ trợ resume và checkpoint `best` được chọn bằng validation balanced accuracy.

> Chọn Kaggle GPU trước khi Run All. ConvNeXt-XXLarge khá nặng; cấu hình mặc định dùng batch vật lý 1 và gradient accumulation 16.

## Kaggle Inputs cần cung cấp

Bắt buộc:

1. Dataset **Tiny-GenImage** có dạng `<generator>/train/{ai,nature}` và `<generator>/val/{ai,nature}`.

Khuyến nghị mạnh:

2. Dataset chứa file `open_clip_pytorch_model.bin` của **laion/CLIP-convnext_xxlarge-laion2B-s34B-b82K-augreg-soup**. Runner tự tìm file này trong `/kaggle/input`. Nếu không cung cấp, bật Internet để tải từ Hugging Face (khoảng 3,4 GB).
3. Dataset cache CommFor đã tạo trước đó gồm `commfor_unseen_manifest.csv` và thư mục `image_cache`. Nếu không có, runner sẽ streaming `OwensLab/CommunityForensics-Eval`, chọn cohort cố định và lưu 1.000 ảnh vào `/kaggle/working`.

Internet phải bật ít nhất ở lần đầu nếu dependencies hoặc pretrained weights chưa nằm trong Kaggle Input. Có thể tạo Kaggle Secret `HF_TOKEN` để tránh giới hạn tải ẩn danh.

In [ ]:
%pip install -q "open_clip_torch==2.26.1" "kornia>=0.7,<0.9" "datasets>=2.19,<4"

In [ ]:
import os
from pathlib import Path

import torch

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    print("HF_TOKEN not configured; public downloads will be anonymous.")

assert torch.cuda.is_available(), "Kaggle Settings → Accelerator → GPU trước khi chạy."
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

In [ ]:
CONFIG = {
    # None = tự tìm Tiny-GenImage trong /kaggle/input.
    "input_root": "/kaggle/input",
    "tiny_dataset_root": None,
    "output_root": "/kaggle/working/aide_full_tiny_commfor",

    # Full run. Paper gốc dùng 20 epochs; đặt 3 để phù hợp phiên Kaggle đơn GPU.
    "epochs": 3,
    "batch_size": 1,
    "gradient_accumulation_steps": 16,
    "num_workers": 2,
    "resume": True,
    "checkpoint_every_optimizer_steps": 250,

    # None = tự tìm open_clip_pytorch_model.bin trong Kaggle Input,
    # rồi mới fallback sang tải từ Hugging Face.
    "semantic_checkpoint": None,

    # Tiny combined.
    "balance_real": True,
    "val_fraction": 0.10,
    "max_train_samples": None,
    "max_val_samples": None,
    "max_tiny_test_samples": None,

    # Fixed CommFor 1,000-image cohort.
    "commfor_manifest_path": None,
    "commfor_allow_hf_fallback": True,
    "commfor_fake_per_generator": 100,
    "commfor_real_per_source": 20,

    # Ảnh cache vẫn nằm trong output; mặc định không nhét ảnh vào ZIP để tải nhanh.
    "package_commfor_images": False,

    # Đặt True để kiểm tra pipeline nhỏ trước; nhớ trả False cho thí nghiệm thật.
    "smoke_test": False,
}

CONFIG

## Bootstrap runtime tự chứa

Cell sau giải nén mã runner đã nhúng trong notebook; không cần upload file `.py` riêng.

In [ ]:
import base64
import gzip
import importlib.util
from pathlib import Path

RUNTIME_GZIP_BASE64 = """..."""
runtime_dir = Path("/kaggle/working/aide_full_runtime")
runtime_dir.mkdir(parents=True, exist_ok=True)
runtime_path = runtime_dir / "train_aide_full_tiny_commfor_kaggle.py"
runtime_path.write_bytes(gzip.decompress(base64.b64decode(RUNTIME_GZIP_BASE64)))

spec = importlib.util.spec_from_file_location("aide_full_runner", runtime_path)
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
print("Runtime:", runtime_path)
print("Full AIDE runner imported successfully.")

## Train và đánh giá

Nếu phiên Kaggle bị dừng nhưng output được lưu, chạy lại cùng `output_root`; runner sẽ đọc checkpoint `latest` và tiếp tục.

In [ ]:
result = runner.run_experiment(CONFIG)
result["experiment"]

## Bảng kết quả và file tải

In [ ]:
import json
import pandas as pd
from IPython.display import FileLink, display

run_dir = Path(CONFIG["output_root"])

def percent_table(path, index_columns):
    frame = pd.read_csv(path)
    metric_columns = [
        "accuracy", "balanced_accuracy", "real_recall", "fake_recall",
        "fake_precision", "fake_f1", "roc_auc", "average_precision",
    ]
    shown = frame[index_columns + ["num_samples", "tn", "fp", "fn", "tp"] +
                  [c for c in metric_columns if c in frame.columns]].copy()
    for column in metric_columns:
        if column in shown:
            shown[column] = (100 * shown[column]).round(2)
    return shown

print("Tiny overall")
display(pd.DataFrame([json.loads((run_dir / "metrics/tiny_overall.json").read_text())]))

print("Tiny per generator (%)")
display(percent_table(run_dir / "metrics/tiny_generator_metrics.csv", ["generator"]))

print("CommFor overall")
display(pd.DataFrame([json.loads((run_dir / "metrics/commfor_overall.json").read_text())]))

print("CommFor per generator: mỗi hàng = 100 fake generator đó + 100 shared real (%)")
display(percent_table(
    run_dir / "metrics/commfor_generator_metrics.csv",
    ["generator", "generator_family"],
))

print("CommFor per family (%)")
display(percent_table(
    run_dir / "metrics/commfor_family_metrics.csv",
    ["generator_family"],
))

archive = Path(result["archive"])
print("Download:")
display(FileLink(str(archive)))

## Output quan trọng

- `checkpoints/best/model_trainable.pt`: checkpoint tốt nhất; không chứa frozen ConvNeXt để tránh thêm khoảng 3,4 GB.
- `checkpoints/latest/model_trainable.pt`: trạng thái resume gồm optimizer/scheduler/scaler.
- `metrics/tiny_generator_metrics.csv`: metric Tiny theo từng generator.
- `metrics/commfor_generator_metrics.csv`: metric CommFor theo từng generator với cùng 100 ảnh real tham chiếu.
- `metrics/commfor_family_metrics.csv`: metric CommFor theo family.
- `predictions/*.csv`: xác suất và dự đoán từng ảnh.
- ZIP cuối notebook chứa checkpoint best, manifests, metrics và predictions; ảnh CommFor cache được giữ ngoài ZIP theo mặc định.